In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Flame_fun import process_flame_dataset

from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF
from sklearn.gaussian_process.kernels import WhiteKernel
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import LeaveOneOut

import warnings

In [ ]:
## Data : Importation + Standardization + Cleaning

csv_path = "Data4ML.csv"
df = pd.read_csv(csv_path, sep=";") 
base_image_folder = r".\data"

X_df = df.iloc[:,:22] 
X_df.PhiInt = X_df.PhiInt / (1 + X_df.PhiInt)

stds = X_df.std()
variables_cst = stds[stds < 1e-5].index.tolist()
X_df = X_df.drop(columns=variables_cst)

variables_corr = ['Xst', 'Zst', 'rho1', 'MW1', 'm1', 'm2'] # Corr with PhiInt 
X_df = X_df.drop(columns=variables_corr)

imgs, mean_flame_image, FL_list = process_flame_dataset(csv_path,base_image_folder,25) 
FL_array = np.array(FL_list).reshape(-1,1)
X = X_df.values
scaler_X_global = StandardScaler()
X_scaled = scaler_X_global.fit_transform(X) 

n_features = X.shape[1]

(94, 1932)


In [ ]:
## Defining the grid search : 

warnings.filterwarnings("ignore")

iso_length = 1.0 # GPR isotrope
ard_length = np.ones(n_features) # GPR anisotrope 

param_grid_rf = {
    # 1. Le nombre d'arbres :
    'n_estimators': [50, 100, 200], 
    
    # 2. La profondeur maximale :
    'max_depth': [3, 5],   
    
    # 3. Le nombre minimum de points physiques pour créer une nouvelle "règle"
    'min_samples_split': [2, 5], 
    
    # 4. Le nombre minimum de points dans une "feuille" finale :
    'min_samples_leaf': [2, 4],   
    
    # 5. Le pourcentage de features regardées à chaque nœud :
    'max_features': [1.0, 'sqrt'] 
}

param_grid_gpr = {
    "kernel": [
        # --- GPR isotropre 
        C(1.0) * Matern(length_scale=iso_length, nu=1.5), # Matern 1.5 : Souple
        C(1.0) * Matern(length_scale=iso_length, nu=2.5), # Matern 2.5 : Intermédiaire 
        C(1.0) * RBF(length_scale=iso_length), # RBF : Très lisse 
        # --- GPR anisotropique 
        C(1.0) * Matern(length_scale=ard_length, nu=1.5),
        C(1.0) * Matern(length_scale=ard_length, nu=2.5),
    ],
    "alpha": [1e-5, 1e-4, 1e-3, 1e-2] 
}

param_grid_svr = {
    'kernel': ['rbf'],
    'C': [0.1, 1, 10, 50, 100, 500],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 0.5],
    'epsilon': [0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.011, 0.012, 0.013, 0.014, 0.015]
}
grid_search_svr = GridSearchCV(SVR(), param_grid_svr, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)


gpr = GaussianProcessRegressor(n_restarts_optimizer=5, normalize_y=True)
rf = RandomForestRegressor(random_state=42)

grid_search_rf = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid_rf, 
    cv=5, 
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1,
    verbose=0)

grid_search_gpr = GridSearchCV(
    estimator=gpr, 
    param_grid=param_grid_gpr, 
    cv=5, 
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1, 
    verbose=0)

grid_search_gpr_res = GridSearchCV(
    estimator=gpr, 
    param_grid=param_grid_gpr, 
    cv=5, 
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1, 
    verbose=0) 


In [ ]:
FL_sqrt = np.sqrt(FL_array)
pls_test = PLSRegression(n_components=8)
pls_test.fit(imgs, FL_sqrt)

y_mode1 = pls_test.x_scores_[:, 0]

kf = KFold(n_splits=10, shuffle=True, random_state=42)

erreurs_gpr = []
erreurs_rf = []
erreurs_hybride = []
erreurs_svr = []
count = 1


for train_idx, test_idx in kf.split(X_scaled):
    print(f"Training on the split {count} / 10 ...")
    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
    y_tr, y_te = y_mode1[train_idx], y_mode1[test_idx]
    
    # --- GPR PUR ---
    print("   Training pure gpr ...")
    grid_search_gpr.fit(X_tr,y_tr)
    pred_gpr = grid_search_gpr.best_estimator_.predict(X_te)
    erreurs_gpr.extend(abs(y_te - pred_gpr))
    
    print("   Training pure SVR ...")
    grid_search_svr.fit(X_tr,y_tr)
    pred_svr = grid_search_svr.best_estimator_.predict(X_te)
    erreurs_svr.extend(abs(y_te - pred_svr))
    
    # --- RF PUR ---
    print("   Training pure rf ...")
    grid_search_rf.fit(X_tr, y_tr)
    pred_rf = grid_search_rf.best_estimator_.predict(X_te)
    erreurs_rf.extend(abs(y_te - pred_rf))
    
    # --- HYBRIDE ---
    print("   Training hyb model rf + gpr ...")
    pred_ref_cv = cross_val_predict(grid_search_rf.best_estimator_, X_tr, y_tr, cv=5)
    residus_tr = y_tr - pred_ref_cv
    
    gpr_res = grid_search_gpr_res.fit(X_tr,residus_tr)
    
    pred_hyb = pred_rf + gpr_res.best_estimator_.predict(X_te)
    erreurs_hybride.extend(abs(y_te - pred_hyb))
    count += 1

# ==========================================
# LES RÉSULTATS
# ==========================================
rmse_svr = np.sqrt(np.mean(np.array(erreurs_svr)**2))
rmse_gpr = np.sqrt(np.mean(np.array(erreurs_gpr)**2))
rmse_rf = np.sqrt(np.mean(np.array(erreurs_rf)**2))
rmse_hybride = np.sqrt(np.mean(np.array(erreurs_hybride)**2))

print("\n" + "="*40)
print("RESULTS")
print("="*40)
print(f"1. Pure GPR       : RMSE = {rmse_gpr:.4f}")
print(f"2. Pure RF        : RMSE = {rmse_rf:.4f}")
print(f"3. Hyb RF+GPR     : RMSE = {rmse_hybride:.4f}")
print(f"4. SVR            : RMSE = {rmse_svr:.4f}")

print("="*40)


Training on the split 1 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 2 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 3 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 4 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 5 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 6 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Training on the split 7 / 10 ...
   Training pure gpr ...
   Training pure SVR ...
   Training pure rf ...
   Training hyb model rf + gpr ...
Traini

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import cross_val_predict, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel


class ResidualStackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, rf_model=None, gpr_model=None, cv=5):
        # On initialise les deux cerveaux
        self.rf_model = rf_model
        self.gpr_model = gpr_model
        self.cv = cv
        
    def fit(self, X, y):
        # Scikit-Learn demande de cloner les modèles en interne
        self.rf_ = clone(self.rf_model)
        self.gpr_ = clone(self.gpr_model)
        
        # Étape 1 : Obtenir les résidus honnêtes
        rf_oof_preds = cross_val_predict(self.rf_, X, y, cv=self.cv)
        residuals = y - rf_oof_preds
        
        # Étape 2 : Entraîner le GPR sur les résidus
        self.gpr_.fit(X, residuals)
        
        # Étape 3 : Entraîner le RF sur toutes les données
        self.rf_.fit(X, y)
        
        return self
        
    def predict(self, X):
        # Étape 4 : La prédiction finale est la somme des deux
        return self.rf_.predict(X) + self.gpr_.predict(X)

# 2. PRÉPARATION DES MODÈLES DE BASE

rf_base = RandomForestRegressor(random_state=42,max_features='sqrt',min_samples_leaf=2)
gpr_base = GaussianProcessRegressor(normalize_y=True, n_restarts_optimizer=3)
modele_hybride = ResidualStackingRegressor(rf_model=rf_base, gpr_model=gpr_base, cv=5)

from sklearn.gaussian_process.kernels import RBF, Matern

# LA GRILLE DE RECHERCHE CONJOINTE MASSIVE
param_grid_conjointe_massive = {
    
    # PARAMÈTRES DU RANDOM FOREST
    'rf_model__max_depth': [3, 4, 5, 6],
    'rf_model__n_estimators': [150, 200, 250],
    
    # PARAMÈTRES DU GPR 
    'gpr_model__kernel': [
        C(1.0) * Matern(length_scale=iso_length, nu=1.5), 
        C(1.0) * Matern(length_scale=iso_length, nu=2.5), 
        C(1.0) * RBF(length_scale=iso_length), 
        C(1.0) * Matern(length_scale=ard_length, nu=1.5),
        C(1.0) * Matern(length_scale=ard_length, nu=2.5),
        C(1.0) * RBF(length_scale=ard_length),
    ], 
    'gpr_model__alpha': [1e-4, 1e-3, 1e-2, 1e-5]
}

print("Lancement de l'optimisation conjointe RF + GPR...")

grid_search_hybride_massif = GridSearchCV(
    estimator=modele_hybride, 
    param_grid=param_grid_conjointe_massive,
    cv=5, 
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, 
    verbose=2
)


grid_search_hybride_massif.fit(X_scaled, y_mode1)

print("\n" + "="*50)
print("MEILLEURE COMBINAISON CONJOINTE TROUVÉE :")
print("="*50)
for param_name, param_value in grid_search_hybride_massif.best_params_.items():
    print(f"{param_name.replace('rf_model__', 'RF - ').replace('gpr_model__', 'GPR - ')} : {param_value}")

meilleur_rmse_cv = -grid_search_hybride_massif.best_score_
print(f"\n RMSE de Validation Croisée : {meilleur_rmse_cv:.4f}")

Lancement de l'optimisation conjointe RF + GPR...
Fitting 5 folds for each of 288 candidates, totalling 1440 fits

🏆 MEILLEURE COMBINAISON CONJOINTE TROUVÉE :
GPR - alpha : 0.01
GPR - kernel : 1**2 * RBF(length_scale=1)
RF - max_depth : 4
RF - n_estimators : 250

🎯 RMSE de Validation Croisée : 9.5644
